### Local view

In [2]:
# # conda install -c conda-forge localtileserver ipyleaflet
# # or:
# # pip install localtileserver ipyleaflet

# from localtileserver import TileClient, get_leaflet_tile_layer
# from ipyleaflet import Map, basemaps, LayersControl

# client = TileClient('../downloaded_data/basal_melt_map_racmo_firn_air_corrected_cog.tif')

# layer = get_leaflet_tile_layer(
#     client,
#     indexes=1,
#     palette="inferno",
#     opacity=0.75,
# )

# m = Map(
#     center=(-80, 0),
#     zoom=2,
#     basemap=basemaps.Esri.WorldImagery,
# )

# m.add_layer(layer)
# m.add_control(LayersControl())
# m

### HV plot

In [4]:
# # conda install -c conda-forge hvplot geoviews datashader dask pyproj
# # or:
# # pip install hvplot geoviews datashader dask pyproj

# import numpy as np
# import rasterio
# import rioxarray
# import cartopy.crs as ccrs
# import hvplot.xarray  # registers .hvplot on xarray objects
# import geoviews as gv
# import geoviews.feature as gf
# from pyproj import Transformer

# gv.extension("bokeh")

# # User inputs
# cog_path = "../downloaded_data/basal_melt_map_racmo_firn_air_corrected_cog.tif"
# overview_level = 3  # choose the COG overview level to display

# # EPSG:3031: Antarctic Polar Stereographic
# proj3031 = ccrs.SouthPolarStereo(
#     central_longitude=0,
#     true_scale_latitude=-71,
# )

# # Map extent equivalent to lon/lat extent [-180, 180, -90, -60]
# to_3031 = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)
# lons = np.linspace(-180, 180, 721)
# x60, y60 = to_3031.transform(lons, np.full_like(lons, -60.0))
# r60 = float(np.nanmax(np.hypot(x60, y60)))

# map_xlim = (-r60, r60)
# map_ylim = (-r60, r60)

# # Open selected overview level
# with rasterio.open(cog_path) as src:
#     num_overviews = len(src.overviews(1))

# print(f"Total overview levels found: {num_overviews}")

# open_kwargs = {
#     "masked": True,
#     "chunks": {"x": 1024, "y": 1024},
# }

# if num_overviews > 0:
#     open_kwargs["overview_level"] = overview_level

# da = rioxarray.open_rasterio(cog_path, **open_kwargs).sel(band=1)
# da.name = "Basal melt"

# print(f"Displaying overview_level={overview_level}")
# print(f"Displayed shape: {da.sizes['y']} x {da.sizes['x']}")

# # Estimate colour limits from a sampled subset
# step_y = max(1, da.sizes["y"] // 2000)
# step_x = max(1, da.sizes["x"] // 2000)

# sample = da.isel(
#     y=slice(None, None, step_y),
#     x=slice(None, None, step_x),
# ).compute()

# vmin, vmax = np.nanpercentile(sample.values, [2, 98])

# # Raster layer
# raster = da.hvplot.quadmesh(
#     x="x",
#     y="y",
#     crs=proj3031,
#     projection=proj3031,
#     rasterize=True,
#     cmap="inferno",
#     clim=(float(vmin), float(vmax)),
#     colorbar=True,
#     clabel="Basal melt",
#     width=800,
#     height=800,
#     xlim=map_xlim,
#     ylim=map_ylim,
#     title=f"COG overview_level={overview_level} | shape={da.sizes['y']} x {da.sizes['x']}",
#     tools=["hover"],
# )

# # Basemap-style context layers
# ocean = gf.ocean.opts(
#     scale="50m",
#     projection=proj3031,
# )

# land = gf.land.opts(
#     scale="50m",
#     projection=proj3031,
#     alpha=0.45,
# )

# coast = gf.coastline.opts(
#     scale="50m",
#     projection=proj3031,
#     line_width=1,
# )

# # Display interactive map
# (ocean * land * raster * coast).opts(
#     active_tools=["wheel_zoom"],
#     toolbar="above",
# )

### OpenLayers COG viewer

In [6]:
# from pathlib import Path
# from html import escape
# from IPython.display import HTML, display

# candidate_paths = [
#     Path("cog_openlayers.html"),
#     Path("4_Visualisation/cog_openlayers.html"),
# ]

# html_path = next((path for path in candidate_paths if path.exists()), None)
# if html_path is None:
#     raise FileNotFoundError("Could not find cog_openlayers.html")

# html_code = html_path.read_text(encoding="utf-8")

# display(HTML(f"""
# <iframe
#     srcdoc="{escape(html_code)}"
#     style="width: 100%; height: 850px; border: 0;"
#     loading="eager">
# </iframe>
# """))